# Ingest generation, load, price, and export data

Use the historical reports plus the API (I hope) to ingest basic electricity data for Alberta

In [0]:
from pyspark import pipelines as dp

In [0]:
from pyspark.sql.functions import col, to_timestamp

catalog_name = spark.conf.get("source_catalog")
schema_name = spark.conf.get("source_schema")
volume_name = spark.conf.get("source_volume")

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/volume_pool-price_ail_historical/"

# Define the streaming table target
dp.create_streaming_table(
    name="pool_price_ail_historical",
    comment="Historical AESO pool price, AIL, generation, and import/export data ingested from CSV"
)

# Append flow for historical CSV ingestion via Auto Loader
@dp.append_flow(
    target="pool_price_ail_historical",
    name="historical_csv_ingest"
)
def historical_csv_ingest():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(volume_path)
        .withColumn("Date_Begin_GMT", to_timestamp(col("Date_Begin_GMT"), "yyyy-MM-dd H:mm"))
        .withColumn("Date_Begin_Local", to_timestamp(col("Date_Begin_Local"), "yyyy-MM-dd H:mm"))
    )